# Requêtes de contrôle NutriScope (TP4, jalon J1)

Cinq contrôles sur la base relationnelle : volumétrie par table, produits sans catégorie,
top marques, complétude Nutri-Score par rayon, doublons restants.

## Prérequis

Avant d'exécuter ce notebook, il faut avoir, dans l'ordre :

1. **Suivi [`docs/postgresql_install.md`](../docs/postgresql_install.md)** : conteneur Docker
   PostgreSQL démarré, base `nutriscope` accessible sur `localhost:5432`.
2. **Créé le schéma** : [`sql/schema.sql`](../sql/schema.sql) exécuté sur la base `nutriscope`.
3. **Généré les extraits Parquet** : `python src/extract_perimeter.py`.
4. **Chargé la base** : `python src/load_database.py`.

## Imports et config nécessaires pour la connexion à la base

In [22]:
import sys
sys.path.append("../src")

from config.settings import POSTGRES_DB_ALIAS
from database.connection import create_duckdb_connection

## Accès à la base

In [23]:
try:
    con = create_duckdb_connection()
    nb_products = con.sql(f"SELECT COUNT(*) FROM {POSTGRES_DB_ALIAS}.public.products").fetchone()[0]
except Exception as error:
    raise ConnectionError(
        "Base nutriscope injoignable : vérifier que le conteneur PostgreSQL tourne "
        "et que le schéma est créé (cf. prérequis)."
    ) from error

if nb_products == 0:
    raise RuntimeError("La table products est vide : lancer python src/load_database.py.")

print(f"Connexion OK, {nb_products} produits en base.")

Connexion OK, 1247309 produits en base.


## 1. Volumétrie par table

In [24]:
con.sql(f"""
    SELECT 'brands' AS table_name, COUNT(*) AS nb_lignes FROM {POSTGRES_DB_ALIAS}.public.brands
    UNION ALL
    SELECT 'categories', COUNT(*) FROM {POSTGRES_DB_ALIAS}.public.categories
    UNION ALL
    SELECT 'products', COUNT(*) FROM {POSTGRES_DB_ALIAS}.public.products
    UNION ALL
    SELECT 'products_categories', COUNT(*) FROM {POSTGRES_DB_ALIAS}.public.products_categories
    UNION ALL
    SELECT 'nutrients', COUNT(*) FROM {POSTGRES_DB_ALIAS}.public.nutrients
    ORDER BY table_name
""").df()

,table_name,nb_lignes
0,brands,79577
1,categories,37442
2,nutrients,1247309
3,products,1247309
4,products_categories,3939635


## 2. Produits sans catégorie

In [25]:
con.sql(f"""
    SELECT COUNT(*) AS nb_produits_sans_categorie
    FROM {POSTGRES_DB_ALIAS}.public.products AS p
    WHERE NOT EXISTS (
        SELECT 1 FROM {POSTGRES_DB_ALIAS}.public.products_categories AS pc WHERE pc.code = p.code
    )
""").df()

,nb_produits_sans_categorie
0,630068


## 3. Top marques (par nombre de produits)

In [26]:
con.sql(f"""
    SELECT b.name AS marque, COUNT(*) AS nb_produits
    FROM {POSTGRES_DB_ALIAS}.public.products AS p
    JOIN {POSTGRES_DB_ALIAS}.public.brands AS b ON b.id = p.brand_id
    GROUP BY b.name
    ORDER BY nb_produits DESC
    LIMIT 10
""").df()

,marque,nb_produits
0,xx:carrefour,11277
1,xx:u,10749
2,xx:auchan,8164
3,xx:casino,6225
4,xx:marque-repere,6217
5,xx:nestle,5898
6,xx:leader-price,5690
7,xx:U,5225
8,xx:le-gaulois,4478
9,xx:cora,4034


## 4. Complétude Nutri-Score par rayon

Le mapping catégories OFF -> 6 rayons n'est pas encore tranché (cf. [`docs/perimetre.md`](../docs/perimetre.md),
risque R-D1 dans [`docs/cadrage/donnees.md`](../docs/cadrage/donnees.md)). Celui-ci est provisoire, basé sur
les tags OFF les plus fréquents du périmètre France.

In [27]:
con.sql(f"""
    WITH rayon AS (
        SELECT
            pc.code,
            CASE
                WHEN c.tag = 'en:beverages' THEN 'Boissons'
                WHEN c.tag = 'en:dairies' THEN 'Produits laitiers'
                WHEN c.tag = 'en:breakfasts' THEN 'Céréales et petit-déjeuner'
                WHEN c.tag IN ('en:biscuits-and-cakes', 'en:salty-snacks') THEN 'Biscuits et snacks'
                WHEN c.tag IN ('en:meals', 'en:canned-foods') THEN 'Plats préparés et conserves'
                WHEN c.tag IN ('en:sauces', 'en:condiments') THEN 'Sauces et condiments'
            END AS rayon
        FROM {POSTGRES_DB_ALIAS}.public.products_categories AS pc
        JOIN {POSTGRES_DB_ALIAS}.public.categories AS c ON c.id = pc.category_id
    ),
    produit_rayon AS (
        -- un produit n'est compté qu'une fois par rayon, même s'il porte plusieurs tags de ce rayon
        SELECT DISTINCT code, rayon FROM rayon WHERE rayon IS NOT NULL
    )
    SELECT
        pr.rayon,
        COUNT(*) AS nb_produits,
        COUNT(*) FILTER (WHERE p.nutriscore_grade IN ('a','b','c','d','e')) AS nb_avec_nutriscore,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.nutriscore_grade IN ('a','b','c','d','e')) / COUNT(*), 1) AS pct_complet
    FROM produit_rayon AS pr
    JOIN {POSTGRES_DB_ALIAS}.public.products AS p ON p.code = pr.code
    GROUP BY pr.rayon
    ORDER BY pct_complet DESC
""").df()

,rayon,nb_produits,nb_avec_nutriscore,pct_complet
0,Biscuits et snacks,55038,50193,91.2
1,Plats préparés et conserves,60005,53843,89.7
2,Produits laitiers,59861,50230,83.9
3,Céréales et petit-déjeuner,33298,22028,66.2
4,Sauces et condiments,37443,23641,63.1
5,Boissons,70702,36861,52.1


## 5. Doublons restants

Doublons stricts sur le code-barres : doit toujours renvoyer 0, la clé primaire de `products` l'interdit.

In [28]:
con.sql(f"""
    SELECT COUNT(*) AS nb_doublons_stricts
    FROM (SELECT code FROM {POSTGRES_DB_ALIAS}.public.products GROUP BY code HAVING COUNT(*) > 1)
""").df()

,nb_doublons_stricts
0,0


In [29]:
con.close()